# Task 3: Player monthly spending (regression, NMAE)

**Goal.** Predict `spending_30d` (THB). Score: `NMAE = 1 / (1 + sum|y - y_hat| / sum y)`. An absolute-error metric is minimised by the **conditional median**, and 48% of the targets are zero, so we model the median directly.

**Final method (as submitted to Kaggle).** Blend of three LightGBM models, 5-fold CV (seed 342, stratified by segment x zero/non-zero):
1. **Two-stage model, raw features** (A): P(spend = 0) from a binary model; log-spend of the positive rows from an L2 model; per-segment lognormal sigma; the conditional median is the prediction (0 if P(zero) >= 0.5).
2. **L1 LightGBM with exact-identity features** (B): `historical_spending = total_transactions x avg_transaction_value` is used to fill missing values; the segment is recovered from the other columns where it is missing.
3. **Two-stage model with the identity features** (C).

Final prediction = clip(0.55 C + 0.25 A + 0.20 B, 0, 300000). **Result:** OOF score about 0.7907. Output: `../outputs/task3_pred.npy`.

In [1]:
import os, re, sys, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

# ---- configuration -------------------------------------------------------------------------------
# DATA = folder of the Kaggle download that holds 01_binary_classification/ ... 05_anomaly_detection/ and sample_submission.csv
DATA = Path(os.environ.get("CPE342_DATA", "")).expanduser()
assert (DATA / "01_binary_classification").exists(), "Set the environment variable CPE342_DATA to the Kaggle download folder (see ../README.md)"
OUT = Path(os.environ.get("CPE342_OUT", "../outputs")); OUT.mkdir(parents=True, exist_ok=True)   # shared by all notebooks
SEED = 342                                   # every random split / model seed in the project
NJOBS = int(os.environ.get("NJOBS", 16))     # LightGBM threads
import lightgbm as lgb
from scipy.stats import norm
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score

def nmae_score(y, pred):
    y = np.asarray(y, float); pred = np.asarray(pred, float)
    return float(1.0 / (1.0 + np.abs(y - pred).sum() / y.sum()))

## 1. Data and quick EDA

In [2]:
d3 = DATA / "03_regression"
tr, te = pd.read_csv(d3 / "train.csv"), pd.read_csv(d3 / "test.csv")
y = tr.spending_30d.values
base_cols = [c for c in tr.columns if c not in ("player_id", "spending_30d")]
print(f"train {len(tr):,} rows, test {len(te):,} rows, {len(base_cols)} features")
print(f"zero share {np.mean(y == 0):.3f} | median {np.median(y):.0f} | mean {y.mean():.0f} | top 1% of rows hold {np.sort(y)[-len(y)//100:].sum() / y.sum():.1%} of the total")
print("zero share by segment:", tr.assign(z=(y == 0)).groupby("segment").z.mean().round(3).to_dict(), "| segment missing:", f"{tr.segment.isna().mean():.1%}")
chk = (tr.historical_spending - tr.total_transactions * tr.avg_transaction_value).abs().dropna()
print(f"historical_spending = total_transactions x avg_transaction_value holds for {np.mean(chk < 1e-6):.1%} of the rows where all three are present")

train 104,000 rows, test 25,889 rows, 32 features
zero share 0.481 | median 79 | mean 10497 | top 1% of rows hold 19.0% of the total
zero share by segment: {0.0: 0.724, 1.0: 0.48, 2.0: 0.306, 3.0: 0.071} | segment missing: 5.7%
historical_spending = total_transactions x avg_transaction_value holds for 99.0% of the rows where all three are present


## 2. Recover the segment and build the identity features
The segment can be predicted from the other columns (cross-validated accuracy about 0.92); the prediction is used only where the segment is missing.

In [3]:
known = tr.segment.notna().values
Xs = tr[[c for c in base_cols if c != "segment"]]
sp = dict(objective="multiclass", num_class=4, learning_rate=0.1, num_leaves=31, min_child_samples=30, verbose=-1, seed=SEED, n_jobs=NJOBS)
idx = np.where(known)[0]; seg_oof = np.full((len(tr), 4), np.nan)
for a, b in KFold(5, shuffle=True, random_state=SEED).split(idx):
    m = lgb.train(sp, lgb.Dataset(Xs.iloc[idx[a]], tr.segment.values[idx[a]].astype(int)), 600)
    seg_oof[idx[b]] = m.predict(Xs.iloc[idx[b]])
print(f"segment recoverable from the other columns: CV accuracy {(seg_oof[idx].argmax(1) == tr.segment.values[idx].astype(int)).mean():.4f}")
mfull = lgb.train(sp, lgb.Dataset(Xs.iloc[idx], tr.segment.values[idx].astype(int)), 600)
seg_tr = tr.segment.copy(); seg_tr[~known] = pd.Series(mfull.predict(Xs).argmax(1), index=tr.index)[~known]
seg_te_hat = pd.Series(mfull.predict(te[[c for c in base_cols if c != "segment"]]).argmax(1), index=te.index)

def fe(df, seg_filled):
    X = df[base_cols].copy()
    tx, av, h, p = df.total_transactions, df.avg_transaction_value, df.historical_spending, df.prev_month_spending
    k = 1.05
    h_f = h.fillna(tx * av).fillna(p * np.exp(k))               # exact identity first, noisy fallback second
    tx_f = tx.fillna(h_f / av.replace(0, np.nan)); av_f = av.fillna(h_f / tx.replace(0, np.nan)); p_f = p.fillna(h_f * np.exp(-k))
    X["hist_f"], X["prev_f"], X["tx_f"], X["avg_f"] = h_f, p_f, tx_f, av_f
    X["log_hist_f"], X["log_prev_f"] = np.log1p(h_f), np.log1p(p_f)
    X["prev_over_hist"] = p / (h + 1)
    X["spend_per_age"] = h_f / (df.account_age_days + 1)
    X["seg_f"] = seg_filled.values
    return X

Xa_fe, Xt_fe = fe(tr, seg_tr), fe(te, te.segment.fillna(seg_te_hat))
Xa_raw, Xt_raw = tr[base_cols], te[base_cols]
strat = (tr.segment.fillna(-1).astype(int).astype(str) + (y > 0).astype(int).astype(str)).values
folds = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(tr, strat))     # the same folds for every model
pos = y > 0
seg_int = tr.segment.fillna(4).astype(int).values; seg_int_te = te.segment.fillna(4).astype(int).values

segment recoverable from the other columns: CV accuracy 0.9207


## 3. Models
`two_stage` returns the conditional median of a zero-inflated lognormal; `l1_model` is a direct median (L1) regression.

In [4]:
def two_stage(Xa, Xt, tag):
    base = dict(learning_rate=0.05, num_leaves=63, min_child_samples=30, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=5.0, verbose=-1, seed=SEED, n_jobs=NJOBS)
    pz = np.zeros(len(Xa)); mu = np.full(len(Xa), np.nan); pz_te = np.zeros(len(Xt)); mu_te = np.zeros(len(Xt))
    for k, (a, b) in enumerate(folds):
        m1 = lgb.train({**base, "objective": "binary"}, lgb.Dataset(Xa.iloc[a], (y[a] > 0).astype(int)), 3000, valid_sets=[lgb.Dataset(Xa.iloc[b], (y[b] > 0).astype(int))], callbacks=[lgb.early_stopping(150, verbose=False)])
        pz[b] = 1 - m1.predict(Xa.iloc[b], num_iteration=m1.best_iteration); pz_te += (1 - m1.predict(Xt, num_iteration=m1.best_iteration)) / 5          # P(y == 0)
        ap, bp = a[pos[a]], b[pos[b]]
        m2 = lgb.train({**base, "objective": "l2"}, lgb.Dataset(Xa.iloc[ap], np.log(y[ap])), 3000, valid_sets=[lgb.Dataset(Xa.iloc[bp], np.log(y[bp]))], callbacks=[lgb.early_stopping(150, verbose=False)])
        mu[b] = m2.predict(Xa.iloc[b], num_iteration=m2.best_iteration); mu_te += m2.predict(Xt, num_iteration=m2.best_iteration) / 5                        # E[log y | y > 0]
        print(f"  [{tag}] fold {k}: zero-AUC {roc_auc_score(y[b] > 0, 1 - pz[b]):.4f} | positive log-RMSE {np.sqrt(np.mean((np.log(y[bp]) - mu[bp]) ** 2)):.4f}", flush=True)
    sig = {s: float(np.std((np.log(y) - mu)[(seg_int == s) & pos])) for s in range(5)}      # per-segment lognormal sigma
    def median_pred(pz_, mu_, sigma_):
        q = np.clip((0.5 - pz_) / np.maximum(1 - pz_, 1e-9), 1e-4, 1 - 1e-4)           # quantile of the positive part that is the overall median
        return np.where(pz_ >= 0.5, 0.0, np.clip(np.exp(mu_ + sigma_ * norm.ppf(q)), 0, 300000))
    oof = median_pred(pz, mu, np.array([sig[s] for s in seg_int])); test = median_pred(pz_te, mu_te, np.array([sig[s] for s in seg_int_te]))
    print(f"  [{tag}] OOF NMAE {nmae_score(y, oof):.5f} | sigma by segment {({s: round(v, 3) for s, v in sig.items()})}")
    return oof, test

def l1_model(Xa, Xt, tag):
    p = dict(objective="l1", metric="l1", learning_rate=0.08, num_leaves=63, min_child_samples=30, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=5.0, verbose=-1, seed=SEED, n_jobs=NJOBS)
    oof, pte = np.zeros(len(tr)), np.zeros(len(Xt))
    for a, b in folds:
        m = lgb.train(p, lgb.Dataset(Xa.iloc[a], y[a]), 8000, valid_sets=[lgb.Dataset(Xa.iloc[b], y[b])], callbacks=[lgb.early_stopping(250, verbose=False)])
        oof[b] = np.clip(m.predict(Xa.iloc[b], num_iteration=m.best_iteration), 0, 300000); pte += np.clip(m.predict(Xt, num_iteration=m.best_iteration), 0, 300000) / 5
    print(f"  [{tag}] OOF NMAE {nmae_score(y, oof):.5f}")
    return oof, pte

oof_A, te_A = two_stage(Xa_raw, Xt_raw, "A two-stage raw")
oof_B, te_B = l1_model(Xa_fe, Xt_fe, "B L1 identity")
oof_C, te_C = two_stage(Xa_fe, Xt_fe, "C two-stage identity")

  [A two-stage raw] fold 0: zero-AUC 0.7837 | positive log-RMSE 0.2199


  [A two-stage raw] fold 1: zero-AUC 0.7834 | positive log-RMSE 0.2292


  [A two-stage raw] fold 2: zero-AUC 0.7844 | positive log-RMSE 0.2277


  [A two-stage raw] fold 3: zero-AUC 0.7836 | positive log-RMSE 0.2183


  [A two-stage raw] fold 4: zero-AUC 0.7874 | positive log-RMSE 0.2242


  [A two-stage raw] OOF NMAE 0.78838 | sigma by segment {0: 0.198, 1: 0.167, 2: 0.165, 3: 0.315, 4: 0.257}


... (3 outputs left out)

  [C two-stage identity] fold 2: zero-AUC 0.7843 | positive log-RMSE 0.2189


  [C two-stage identity] fold 3: zero-AUC 0.7841 | positive log-RMSE 0.2159


  [C two-stage identity] fold 4: zero-AUC 0.7867 | positive log-RMSE 0.2183


  [C two-stage identity] OOF NMAE 0.78995 | sigma by segment {0: 0.182, 1: 0.165, 2: 0.159, 3: 0.315, 4: 0.22}


## 4. Blend and output

In [5]:
W = dict(C=0.55, A=0.25, B=0.20)
oof_bl = np.clip(W["C"] * oof_C + W["A"] * oof_A + W["B"] * oof_B, 0, 300000)
te_bl = np.clip(W["C"] * te_C + W["A"] * te_A + W["B"] * te_B, 0, 300000)
print(f"OOF NMAE: A {nmae_score(y, oof_A):.5f} | B {nmae_score(y, oof_B):.5f} | C {nmae_score(y, oof_C):.5f} | blend {nmae_score(y, oof_bl):.5f}   (earlier blend 0.5 A + 0.5 B: {nmae_score(y, np.clip(.5 * oof_A + .5 * oof_B, 0, 300000)):.5f})")
err = np.abs(y - oof_bl); print("absolute-error share by segment:", {s: round(err[seg_int == s].sum() / err.sum(), 3) for s in range(5)}, "(4 = segment missing)")
for n, a in (("two_raw", te_A), ("l1_identity", te_B), ("two_identity", te_C)): np.save(OUT / f"task3_test_{n}.npy", a)     # members, kept for the alternative final file (V53c)
np.save(OUT / "task3_oof_blend.npy", oof_bl); np.save(OUT / "task3_pred.npy", te_bl)
print(f"test prediction: mean {te_bl.mean():.1f}, zeros {int((te_bl == 0).sum())} | saved ../outputs/task3_pred.npy")

OOF NMAE: A 0.78838 | B 0.78634 | C 0.78995 | blend 0.79071   (earlier blend 0.5 A + 0.5 B: 0.78962)
absolute-error share by segment: {0: np.float64(0.01), 1: np.float64(0.058), 2: np.float64(0.075), 3: np.float64(0.796), 4: np.float64(0.06)} (4 = segment missing)
test prediction: mean 10435.3, zeros 39 | saved ../outputs/task3_pred.npy
